# Oracles Workbook

What is this workbook? A workbook is a collection of problems, accompanied by solutions to them. The explanations focus on the logical steps required to solve a problem; they illustrate the concepts that need to be applied to come up with a solution to the problem, explaining the mathematical steps required.

This workbook describes the solutions to the problems offered in the "Oracles" kata. Since the problems involve code implementations of the solutions, the explanations also cover some elements of Workbench that might be non-obvious for a first-time user.

In [1]:
# Execute this cell to prepare the test infrastructure
from psiqdk.workbench import QPU, Qubits


## Problem 1. Implement a classical oracle

The binary notation of $7$ is $111_2$. This means that your solution should return `True` if and only if each bit of the input is `True`.

Notice that in this problem the solution doesn't depend on whether the bit string is converted into an integer using big-endian or little-endian notation. In general, this is an important distinction that you need to take into account.

In [2]:
def is_seven(x: list[bool]) -> bool:
    return x[0] and x[1] and x[2]

## Problem 2. Implement a phase oracle for marking 7

You can flip the phase of the basis state $\ket{111}$ using a controlled $Z$ gate, with any two of the qubits as controls and the third one as the target.

In [3]:
def is_seven_phase_oracle(x: Qubits) -> None:
    x[0].z(cond=x[1:])

Consider how this oracle acts on two basis states:
$$U_{7,phase} \ket{111} = -\ket{111}$$
$$U_{7,phase} \ket{110} = \ket{110}$$

You can see that $U_{7,phase}$ doesn't change the input if it's a basis state (other than adding a global phase), and $U_{7,phase}$ doesn't change the norm of the state ($U_{7,phase}$ is a unitary operator).  

However, if you applied this oracle to a superposition state instead, what will that look like?

Suppose that $\ket{\beta}$ is an equal superposition of the states  $\ket{110}$ and $\ket{111}$: 
$$\ket{\beta} = \tfrac{1}{\sqrt{2}} \big(\ket{110} + \ket{111}\big) = \ket{11} \otimes \tfrac{1}{\sqrt{2}} \big(\ket{0} + \ket{1}\big) = \ket{11} \otimes \ket{+} = \ket{11+}$$

Let's consider how the operator $U_{7,phase}$ acts on this state:

$$U_{7,phase} \ket{\beta} = U_{7,phase} \Big[\tfrac{1}{\sqrt{2}} \big(\ket{110} + \ket{111}\big)\Big] =$$

$$= \tfrac{1}{\sqrt{2}} \big(U_{7,phase} \ket{110} + U_{7,phase} \ket{111}\big) =$$

$$= \tfrac{1}{\sqrt{2}} \big(\ket{110} - \ket{111}\big) := \ket{\gamma}$$

Was your input state modified during this operation? Let's simplify $\ket{\gamma}$:

$$\ket{\gamma} = \tfrac{1}{\sqrt{2}} \big(\ket{110} - \ket{111}\big) = \ket{11} \otimes \tfrac{1}{\sqrt{2}} \big(\ket{0} - \ket{1}\big) = \ket{11} \otimes \ket{-} = \ket{11-} \neq \ket{\beta}$$

Here you see that the oracle modifies the input, if the input state is a *superposition* of the basis states, since a phase oracle will only modify the sign of the basis states.

> It's also worth noting that while the oracle modified the input when provided a superposition state, it did *not* modify the norm of that state.  As an exercise, you can verify this yourself by taking the norm of $\ket{\beta}$ and $\ket{\gamma}$, which both will result in a value of $1$.
>
> As another exercise, consider how you could distinguish between the input and output state programmatically?  Is there an operation that you could apply to the initial state $\ket{\beta}$ and the final state $\ket{\gamma}$ to show that the two states aren't equivalent through measurement?  As a hint, think about how you could convert the superposition states $\ket{\beta}$ and $\ket{\gamma}$ into the basis states.

## Problem 3. Implement a marking oracle for marking 7

You can flip the state of the target qubit if the basis state of the control qubits is $\ket{111}$ using a controlled $X$ gate.

In [4]:
def is_seven_marking_oracle(x: Qubits, y: Qubits) -> None:
    y.x(cond=x == 7)

Consider how the oracle from this exercise acts on two input basis states and two "output" basis states:

$$U_{7,mark} \ket{111} \ket{0} = \ket{111} \ket{0 \oplus f(111)} = \ket{111} \ket{0 \oplus 1} = \ket{111} \ket{1}$$

$$U_{7,mark} \ket{111} \ket{1} = \ket{111} \ket{1 \oplus f(111)} = \ket{111} \ket{1 \oplus 1} = \ket{111} \ket{0}$$

$$U_{7,mark} \ket{110} \ket{0} = \ket{110} \ket{0 \oplus f(110)} = \ket{110} \ket{0 \oplus 0} = \ket{110} \ket{0}$$

$$U_{7,mark} \ket{110} \ket{1} = \ket{110} \ket{1 \oplus f(110)} = \ket{110} \ket{1 \oplus 0} = \ket{110} \ket{1}$$

You can see that the state of the input qubit register is unchanged, and the state of the output qubit changes if $f(x) = 1$ and is unchanged if $f(x) = 0$ - this matches the definition of a marking oracle precisely.

Now let's again apply this oracle to a superposition state $\ket{\alpha}$ such that $\ket{x}$ is a superposition of the basis states $\ket{110}$ and $\ket{111}$ and $\ket{y} = \ket{0}$:
$$\ket{\alpha} = \tfrac{1}{\sqrt{2}}\big(\ket{110} + \ket{111}\big)\ket{0} = 
\ket{11} \otimes \tfrac{1}{\sqrt{2}} \big(\ket{0} + \ket{1}\big) \otimes \ket{0} = \ket{11+} \ket{0}$$

Let's consider how the operator $U_{7,mark}$ acts on this state.

> Recall that oracles are linear operators, thus they can be applied to each term individually.

$$U_{7,mark} \ket{\alpha} = \tfrac{1}{\sqrt{2}} \big(U_{7,mark}\ket{110} \ket{0} + U_{7,mark}\ket{111} \ket{0}\big) =$$

$$= \tfrac{1}{\sqrt{2}} \big(\ket{110} \ket{0} + \ket{111} \ket{1}\big) := \ket{\epsilon}$$

Was your input state modified during this operation?  Let's simplify the resulting state $\ket{\epsilon}$:

$$\ket{\epsilon} = \tfrac{1}{\sqrt{2}} \big(\ket{110} \ket{0} + \ket{111} \ket{1}\big) = \ket{11} \otimes \tfrac{1}{\sqrt{2}} \big(\ket{0} \ket{0} + \ket{1} \ket{1}\big) =$$

$$= \ket{11} \otimes \tfrac{1}{\sqrt{2}} \big(\ket{00} + \ket{11}\big) = \ket{11} \otimes \ket{\Phi^+} = \ket{11\Phi^+}$$

You have entangled the states of qubits $\ket{x}$ and $\ket{y}$!  This is a common occurrence for marking oracles when the input is a superposition of basis states: after applying the oracle, the input $\ket{x}$ will often become entangled with $\ket{y}$. Thus, while applying the marking oracle to a basis state will leave the input register unchanged, applying the marking oracle to a superposition state will change the state of both the input register and the output qubit.

>As an exercise, what entangled state would you get in the previous example if $\ket{y} = \ket{1}$ instead of $\ket{y} = \ket{0}$?
>
> <details>
>   <summary><b>Answer</b></summary>
>
> $$U_{7,mark} \ket{11+} \ket{1} = \ket{11} \otimes \tfrac1{\sqrt2}\big(\ket{01} + \ket{10}\big) = \ket{11} \ket{\Psi^+}$$
> </details>

## Problem 4. Implement a marking oracle as a phase oracle

As you just saw, you can allocate an additional qubit in the $\ket{-}$ state and use it as the "output" for our marking oracle.
This will kick back the $-1$ relative phase for the basis states $\ket{x}$ of the input register for which $f(x) = 1$.

In [5]:
def apply_marking_oracle_as_phase_oracle(marking_oracle: callable, x: Qubits) -> None:
    y = Qubits(1, "y", x.qpu)

    # Create the minus state |-⟩
    y.x()
    y.had()

    # Apply the marking Oracle
    marking_oracle(x, y)

    # Return the auxiliary qubit to the zero state
    y.had()
    y.x()

    # Release the auxiliary qubit
    y.release()

## Problem 5. Implement the OR oracle

What if you needed to flip the state of the output qubit only if the input register is in the $\ket{0...0}$ state? In that case, you could apply an $X$ gate to the output qubit with the condition `x == 0` (register `x` with all qubits in $\ket{0}$ state).

In this problem, you need to separate the $\ket{0...0}$ basis state from all others, but with the opposite effect on the output qubit: instead of flipping it for this input state, you need to flip it for all other input states. Or, you can think of it as first flipping the state of the target qubit for all states, and then un-flipping it (or flipping it again) for just this basis state. You can do this by applying an unconditional $X$ gate before or after the conditional $X$ gate.

In [ ]:
def or_oracle(x: Qubits, y: Qubits) -> None:
    y.x()
    y.x(cond=x == 0)

## Problem 6. Implement the K-th bit oracle

Since the effect of this oracle depends only on the value of the $k$-th qubit, you can ignore the rest of the qubits and focus on just `x[k]`. You need to flip the phase of the qubit if it is in the $\ket{1}$ state and leave it unchanged otherwise - this is exactly the effect of the $Z$ gate.

In [7]:
def kth_bit_oracle(x: Qubits, k: int) -> None:
    x[k].z()

## Problem 7. Implement the OR oracle of all bits except the K-th

The easiest way to solve this task is to build upon your implementation of the marking OR oracle from an earlier problem and use the phase kickback trick to convert it into a phase oracle.

Since in this task you're evaluating OR of all bits except the $k$-th one, you need to exclude the qubit `x[k]` from the list of control qubits for the marking oracle. You can do that using slicing: `x[:k]` gets the subregister of qubits before the $k$-th one, and `x[k + 1:]` gets the subregister of qubits after the $k$-th one. Your register of control qubits will be a concatenation of these two arrays: `x[:k] | x[k+1:]`. Notice, however, that Workbench doesn't allow a Qubits register slice to have no qubits in it; you'll need to handle the cases of $k = 0$ and $k = N - 1$ separately.

In [8]:
def or_of_bits_except_kth_oracle(x: Qubits, k: int) -> None:
    y = Qubits(1, "y", x.qpu)

    y.x()
    y.had()

    # Handle k being first/last qubit of the register
    if k == 0:
        or_oracle(x[1:], y)
    elif k == len(x[:])-1:
        or_oracle(x[:-1], y)
    else:
        or_oracle(x[:k] | x[k+1:], y)

    y.had()
    y.x()

    y.release()

## Problem 8. Implement the arbitrary bit pattern oracle

This oracle effectively applies a controlled $X$ gate on the output qubit `y` if the control qubits `x` are in a certain state. In Workbench, you can do that using the argument `cond`. You need to convert the bit pattern to an integer (in little-endian encoding) and to pass the result of the comparison as the argument `cond` to the `x()` method you call on the register `y`.

In [9]:
def list_to_int(x: list[bool]) -> int:
    val = 0
    for i, x_i in enumerate(x):
        val += int(x_i) * 2 ** i
    return val


def arbitrary_bit_pattern_oracle(x: Qubits, y: Qubits, pattern: list[bool]) -> None:
    k = list_to_int(pattern)
    y.x(cond=x == k)

## Problem 9. Implement the arbitrary bit pattern oracle (challenge version)

The most straightforward solution is to transform the given state so that the basis state that matches the given pattern becomes $\ket{1...1}$, apply the Controlled $Z$ gate to flip the phase of just that basis state, and then uncompute to make sure that only the relative phases of the basis states change, not the basis states themselves.

In [10]:
def arbitrary_bit_pattern_oracle_challenge(x : Qubits, pattern : list[bool]) -> None:
    for i in range(len(pattern)):
        if not pattern[i]:
            x[i].x()

    x[0].z(cond=x[1:])

    for i in range(len(pattern)):
        if not pattern[i]:
            x[i].x()

You can express the same transformation more concisely using the `reflect` method of the Qubits class. This method multiplies one basis state by a given relative phase, leaving others unchanged. By default, calling `reg.reflect()` will multiply the $\ket{1...1}$ basis state by $-1$. Calling `(reg == p).reflect()` will multiply the basis state $\ket{p}$ by $-1$ - and this is exactly what we want to do, if $p$ is the integer representation of `pattern`. 

In [11]:
def arbitrary_bit_pattern_oracle_challenge(x: Qubits, pattern: list[bool]) -> None:
    p = sum([pattern[i] * 2 ** i for i in range(x.num_qubits)])
    (x == p).reflect()

> Copyright (c) 2026 PsiQuantum